# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id`.

### Dataset Source
The dataset is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published on: {getattr(metadata, 'datePublished', 'Unknown')}")

## 2. Data Overview
Review available record sets and their fields, referencing all entities by their Croissant `@id`.


In [ ]:
# List all record sets by `@id` and show their fields and columns
def get_record_sets(meta):
    # The Croissant v1 structure stores recordSet as a list under 'recordSet'
    if hasattr(meta, 'recordSet'):
        record_sets = meta.recordSet
        if isinstance(record_sets, dict):  # Croissant sometimes non-list
            record_sets = [record_sets]
        return record_sets
    return []

record_sets = get_record_sets(metadata)

print("Available Record Sets (@id):\n--------------------------")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', 'no name')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):  # single field
        fields = [fields]
    for f in fields:
        print(f"   - Field @id: {f['@id']} | Name: {f.get('name', 'no name')} | DataType: {f.get('dataType', 'unknown')}")
        for col in f.get('column', []) if isinstance(f.get('column', []), (list, tuple)) else [f.get('column', None)]:
            if col is not None:
                print(f"     - Column @id: {col['@id']} | Name: {col.get('name', 'no name')}")

## 3. Data Extraction
Load records from a selected record set into a DataFrame for analysis. We will use one of the main record set `@id`s from the overview above as an example. Adjust `<record_set_id>` as needed for your analysis.

In [ ]:
# Identify main record set for protein analysis. (Replace @id below with a real one from section 2)
main_record_set_id = None
if record_sets:
    # Heuristic: First record set, or match by name
    for rs in record_sets:
        if 'protein' in (rs.get('name', '') + rs['@id']).lower() or 'results' in (rs.get('name', '') + rs['@id']).lower():
            main_record_set_id = rs['@id']
            break
    if not main_record_set_id:
        main_record_set_id = record_sets[0]['@id']  # Fallback to first
else:
    raise Exception("No record sets found in metadata.")

print(f"Selected main record set: {main_record_set_id}\n")

df_records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(df_records)
print(f"Columns loaded from record set {main_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply some common data processing and analysis tasks. We reference field and column variables using their Croissant `@id`s where possible.

In [ ]:
# Choose a numeric field for EDA, for example 'Abundance' or 'MW' (molecular weight)
# Find a numeric field from the DataFrame columns (heuristic)
numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32] or all(pd.to_numeric(df[col], errors='coerce').notnull())]
print('Numeric field candidates:', numeric_candidates)
numeric_field = None
for cand in ['MW', 'Abundance', 'coverage', 'count', 'peptide']:
    for col in df.columns:
        if cand.lower() in col.lower():
            numeric_field = col
            break
    if numeric_field:
        break
if not numeric_field and len(numeric_candidates) > 0:
    numeric_field = numeric_candidates[0]
if not numeric_field:
    raise Exception("No suitable numeric field found.")
print(f"Using numeric field: {numeric_field}")
# Ensure the numeric column is float
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = np.nanquantile(df[numeric_field], 0.9)  # Top 10% as example
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.3f}  (Top 10%)")
print(filtered_df.head())
# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
# Try grouping by a categorical field, e.g., 'description'
group_field = None
for cand in ['description', 'accession', 'protein']:
    for col in df.columns:
        if cand.lower() in col.lower():
            group_field = col
            break
    if group_field:
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships. Here we show the distribution of the selected numeric field, and a boxplot by group if available.

In [ ]:
plt.figure(figsize=(8,4))
df[numeric_field].dropna().hist(bins=30, alpha=0.7)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

if group_field is not None:
    # Show boxplot by group (showing only up to 10 unique values for clarity)
    groups = df[group_field].value_counts().head(10).index
    plt.figure(figsize=(10,5))
    box = df[df[group_field].isin(groups)].boxplot(column=numeric_field, by=group_field, grid=False, rot=60)
    plt.title(f"{numeric_field} by {group_field}")
    plt.suptitle('')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion

- We loaded the FAIR² mass spectrometry protein dataset using `mlcroissant` via its Croissant schema URL.
- We examined the available record sets and fields using their Croissant `@id`s.
- We extracted tabular data from the main record set, identified numeric fields, filtered and normalized protein abundance or MW, and grouped by descriptive fields.
- Finally, we visualized data distributions and group statistics.

For further analysis, explore additional record sets, fields, or perform domain-specific protein network/pathway investigations using the extracted data. Always refer to dataset documentation and entity `@id`s for reproducibility.